# SQL子查询（习题）

In [1]:
import duckdb

%load_ext sql
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

In [2]:
%%sql
CREATE OR REPLACE VIEW sales AS SELECT * FROM '../data/sales.csv'; 

Running query in 'duckdb:///:memory:'

Count


Easy

1. (回收 Week 1 第 12 题)找出金额(total)高于「全表平均金额」的所有订单,输出 order_id, total,按金额降序。要求用标量子查询。

In [3]:
%%sql
SELECT order_id, total
FROM sales
WHERE total > (
    SELECT AVG(total) AS avg_total
    FROM sales
)
ORDER BY total DESC;

Running query in 'duckdb:///:memory:'

order_id,total
O1110,9995
O1238,9995
O1470,9995
O1409,9995
O1152,9995
O1483,9995
O1432,9995
O1319,9995
O1324,9995
O1342,9995


参考答案

逻辑完全对。一个细节:子查询里的 AS avg_total 没有意义。标量子查询返回的就是一个值,外层 WHERE total > (...) 根本不会去引用这个别名。别名只在「结果要被别人按名字引用」时才有用。这里删掉更干净:

In [37]:
# WHERE total > (SELECT AVG(total) FROM sales)

2. 找出金额正好等于「全表最大金额」的订单。 思考:为什么这题不能简单 ORDER BY total DESC LIMIT 1?如果有并列最大值会怎样?

In [ ]:
%%sql
SELECT order_id, total
FROM sales
WHERE total = (
    SELECT MAX(total) AS max_total
    FROM sales
);

Running query in 'duckdb:///:memory:'

order_id,total
O1033,9995
O1110,9995
O1152,9995
O1228,9995
O1238,9995
O1319,9995
O1324,9995
O1342,9995
O1355,9995
O1370,9995


3. 先查出「订单数最多的那个国家」,再列出该国家的全部订单。允许嵌套子查询。

In [14]:
%%sql
SELECT order_id, country
FROM sales 
WHERE country = (
    SELECT country
    FROM sales 
    GROUP BY country
    ORDER BY COUNT(*) DESC
    LIMIT 1
)

Running query in 'duckdb:///:memory:'

order_id,country
O1005,UK
O1006,UK
O1007,UK
O1008,UK
O1010,UK
O1012,UK
O1013,UK
O1019,UK
O1023,UK
O1025,UK


Medium

4. 对每个订单,在结果里多带两列:它所属 category(品类)的平均金额、以及本订单金额是「高」还是「低」于该品类平均(输出字符串 '高' / '低')。用相关子查询。
(注意:这里换成按 category 分组,不是 country——逼你别照抄跟练代码。)

In [ ]:
%%sql
SELECT s1.*,
       (SELECT AVG(total) 
        FROM sales s2
        WHERE s2.category = s1.category 
        GROUP BY category
        ) AS avg_category_price,
FROM sales s1

Running query in 'duckdb:///:memory:'

order_id,customer_id,product,category,quantity,price,order_date,country,total,avg_category_price
O1000,C007,Keyboard,Accessory,2,1299,2024-01-01,Germany,2598,2451.4777777777776
O1001,C004,Keyboard,Accessory,1,99,2024-01-01,US,99,2451.4777777777776
O1002,C005,Laptop,Computer,4,99,2024-01-02,US,396,2274.58125
O1003,C007,Headphones,Audio,4,99,2024-01-03,US,396,3055.0149253731342
O1004,C003,Phone,Mobile,5,99,2024-01-03,France,495,2772.3763440860216
O1005,C008,Laptop,Computer,2,99,2024-01-04,UK,198,2274.58125
O1006,C005,Phone,Mobile,5,1299,2024-01-05,UK,6495,2772.3763440860216
O1007,C005,Monitor,Computer,2,599,2024-01-06,UK,1198,2274.58125
O1008,C007,Phone,Mobile,5,299,2024-01-06,UK,1495,2772.3763440860216
O1009,C002,Headphones,Audio,3,99,2024-01-07,Germany,297,3055.0149253731342


输出高/低的内容根本就没学过，以后不要出超纲的内容了

参考答案

问题 1(真 bug):子查询里的 GROUP BY category 是错的、要删掉。

相关子查询已经靠 WHERE s2.category = s1.category 把数据锁定到当前这一个品类了,此时 AVG(total) 算的就是这一个品类的平均。再加 GROUP BY category 属于画蛇添足——逻辑上你已经只剩一个品类,分组没有意义,还可能让子查询「返回多行」而违反标量子查询「只能返回 1 行 1 列」的规定。删掉它。

问题 2:avg_category_price, 末尾多了一个逗号。
FROM 前面那一列后面跟了逗号,DuckDB 容忍,PostgreSQL 直接报错。这是你弱点清单 #17 原话——「SELECT 最后一列后面不要加逗号」,今天复发了。

问题 3:列名词不对。 你算的是 AVG(total)(金额),却命名 avg_category_price。total 是订单总额,不是 price。命名要诚实。

In [38]:
%%sql
SELECT
    s1.order_id,
    s1.category,
    s1.total,
    (SELECT AVG(s2.total)
     FROM sales s2
     WHERE s2.category = s1.category) AS avg_category_total,
    CASE
        WHEN s1.total > (SELECT AVG(s2.total)
                         FROM sales s2
                         WHERE s2.category = s1.category)
        THEN '高' ELSE '低'
    END AS 高低
FROM sales s1;

Running query in 'duckdb:///:memory:'

order_id,category,total,avg_category_total,高低
O1000,Accessory,2598,2451.4777777777776,高
O1001,Accessory,99,2451.4777777777776,低
O1002,Computer,396,2274.58125,低
O1003,Audio,396,3055.0149253731342,低
O1004,Mobile,495,2772.3763440860216,低
O1005,Computer,198,2274.58125,低
O1006,Mobile,6495,2772.3763440860216,高
O1007,Computer,1198,2274.58125,低
O1008,Mobile,1495,2772.3763440860216,低
O1009,Audio,297,3055.0149253731342,低


5. 找出「消费总额高于全体客户平均消费总额」的客户及其消费总额,按消费总额降序。用 FROM 派生表。

做的时候留意:你会发现要把同一段子查询写两遍——在笔记里记下这个痛点。

In [19]:
%%sql
SELECT customer_id, customer_total
FROM (
    SELECT customer_id, SUM(total) AS customer_total
    FROM sales
    GROUP BY customer_id
) AS t
WHERE customer_total > (
    SELECT AVG(customer_total)
    FROM(
        SELECT SUM(total) AS customer_total
        FROM sales
        GROUP BY customer_id
    ) AS t2
)
ORDER BY customer_total DESC;

Running query in 'duckdb:///:memory:'

customer_id,customer_total
C006,198806
C001,184001
C008,164484
C003,160212


6. 用 IN 子查询:找出「购买过数量 ≥ 5 的订单」的那些客户,列出这些客户的全部订单(包括他们 quantity < 5 的订单)。

In [22]:
%%sql
SELECT *
FROM sales
WHERE customer_id IN(
    SELECT customer_id
    FROM sales
    WHERE quantity >= 5
)

Running query in 'duckdb:///:memory:'

order_id,customer_id,product,category,quantity,price,order_date,country,total
O1000,C007,Keyboard,Accessory,2,1299,2024-01-01,Germany,2598
O1001,C004,Keyboard,Accessory,1,99,2024-01-01,US,99
O1002,C005,Laptop,Computer,4,99,2024-01-02,US,396
O1003,C007,Headphones,Audio,4,99,2024-01-03,US,396
O1004,C003,Phone,Mobile,5,99,2024-01-03,France,495
O1005,C008,Laptop,Computer,2,99,2024-01-04,UK,198
O1006,C005,Phone,Mobile,5,1299,2024-01-05,UK,6495
O1007,C005,Monitor,Computer,2,599,2024-01-06,UK,1198
O1008,C007,Phone,Mobile,5,299,2024-01-06,UK,1495
O1009,C002,Headphones,Audio,3,99,2024-01-07,Germany,297


IN 的语法规则（死规定）：

左边是 1 个字段（customer_id）

右边必须是 只有 1 列的结果

你写 SELECT * → 返回多列 → 直接报错

7. 用 EXISTS:找出「从来没有下过金额 > 300 订单」的客户。 先想清楚:该用 EXISTS 还是 NOT EXISTS?

In [23]:
%%sql
SELECT customer_id
FROM sales s1
WHERE NOT EXISTS (
    SELECT 1
    FROM sales s2
    WHERE s2.customer_id = s1.customer_id
    AND s2.total > 300
)

Running query in 'duckdb:///:memory:'

customer_id


参考答案

NOT EXISTS 用对了(「从来没下过 > 300 的订单」=「不存在 > 300 的订单」→ NOT EXISTS,你判断对了)。

但 bug 在 FROM sales:sales 是订单表,一个客户有多少笔订单,customer_id 就会出现多少次。比如客户 C003 有 8 笔订单且都 ≤ 300,你的结果里 C003 会重复出现 8 次。

题目要的是「客户」,客户应该是唯一的。两个修法:

In [39]:
%%sql
-- 方法 A:加 DISTINCT
SELECT DISTINCT customer_id
FROM sales s1
WHERE NOT EXISTS (
    SELECT 1
    FROM sales s2
    WHERE s2.customer_id = s1.customer_id
    AND s2.total > 300
)

Running query in 'duckdb:///:memory:'

customer_id


In [40]:
%%sql
-- 方法 B(更本质):从"客户列表"出发,而不是从"订单表"出发
SELECT customer_id
FROM (SELECT DISTINCT customer_id FROM sales) s1
WHERE NOT EXISTS (
    SELECT 1 FROM sales s2
    WHERE s2.customer_id = s1.customer_id AND s2.total > 300
)

Running query in 'duckdb:///:memory:'

customer_id


Hard

8. 找出每个国家金额最高的那一笔订单(每个国家一行,若并列则都保留),输出 country, order_id, total。用相关子查询实现 "top-1 per group"——提示:WHERE total = (该国家的 MAX(total))。

In [ ]:
%%sql
SELECT s1.country, s1.order_id, s1.total
FROM sales s1
WHERE s1.total = (
    SELECT MAX(s2.total)
    FROM sales s2
    WHERE s2.country = s1.country
)
ORDER BY s1.country

Running query in 'duckdb:///:memory:'

country,order_id,total
China,O1035,6495
China,O1237,6495
France,O1319,9995
France,O1483,9995
Germany,O1342,9995
Germany,O1370,9995
UK,O1033,9995
UK,O1110,9995
UK,O1152,9995
UK,O1228,9995


9. 先算出每个月的 GMV(月度总销售额),再找出「GMV 高于全年月均 GMV」的月份,按月份排序。
月份用 STRFTIME(order_date, '%Y-%m')。你的 order_date 是 DATE 类型,STRFTIME 可以直接用——呼应你 Day 6 笔记「日期处理别用 SUBSTRING」。

In [ ]:
%%sql
SELECT STRFTIME(order_date, '%Y-%m') AS order_month,
        SUM(total) AS total_sales
FROM sales
GROUP BY order_month
HAVING total_sales > (
    SELECT AVG(total_sales)
    FROM (
        SELECT STRFTIME(order_date, '%Y-%m') AS order_month,
               SUM(total) AS total_sales
        FROM sales
        GROUP BY order_month
    ) AS monthly_sales
)
ORDER BY order_month;

Running query in 'duckdb:///:memory:'

order_month,total_sales
2024-03,133371
2024-04,123882
2024-06,119061
2024-10,117265
2024-11,105984
2024-12,106975


10. —(NULL 陷阱实战) 自己构造一份带 NULL 的小数据:用 VALUES 造一张 vip 表,含客户 id (101, 102, NULL)。然后:

• (a) 用 NOT IN 写「不在 vip 表里的客户」,从 sales 里查,观察结果

• (b) 用 NOT EXISTS 写同样的需求

• (c) 在笔记里写清楚两者结果的差异和原因

In [31]:
%%sql
CREATE OR REPLACE TABLE demo AS
    SELECT * FROM (VALUES (101), (102), (103)) AS t(id);

CREATE OR REPLACE TABLE blocked AS
    SELECT * FROM (VALUES (101), (102), (NULL)) AS t(id);

Running query in 'duckdb:///:memory:'

Count


In [33]:
%%sql
SELECT id FROM demo WHERE id NOT IN (SELECT id FROM blocked);

Running query in 'duckdb:///:memory:'

id


In [36]:
%%sql
SELECT d.id FROM demo d WHERE NOT EXISTS (SELECT 1 FROM blocked b WHERE b.id = d.id);

Running query in 'duckdb:///:memory:'

id
103


NOT EXISTS 不受 NULL 影响而IN会受到。